In [1]:
import subprocess
import re
import itertools


In [2]:
alpha_values = [0.1, 0.3, 0.5, 0.7, 0.9]
beta_values  = [0.1, 0.3, 0.5, 0.7, 0.9]
gamma_values = [0.1, 0.3, 0.5, 0.7, 0.9]

In [3]:
# Store results
results = []

def run_test(alpha, beta, gamma):
    # Build and source environment
    subprocess.run(["bash", "./build.sh"], check=True)
    subprocess.run(["bash", "-c", "source devel/setup.bash"], shell=True, check=True)

    # Set ROS parameters for BRRT_Optimize
    subprocess.run(["rosparam", "set", "/BRRT_Optimize/alpha", str(alpha)], check=True)
    subprocess.run(["rosparam", "set", "/BRRT_Optimize/beta", str(beta)], check=True)
    subprocess.run(["rosparam", "set", "/BRRT_Optimize/gamma", str(gamma)], check=True)

    # Run the test launch file and capture output
    proc = subprocess.run(
        ["roslaunch", "path_finder", "test_planners.launch"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=300
    )
    output = proc.stdout

    # Parse metrics using regex (adjust patterns to your log format)
    brrt_match = re.search(r"BRRT.*length:\s*([\d\.]+)", output)
    opt_match  = re.search(r"BRRT_Optimize.*length:\s*([\d\.]+)", output)

    if brrt_match and opt_match:
        brrt_length = float(brrt_match.group(1))
        opt_length  = float(opt_match.group(1))
        return brrt_length, opt_length
    else:
        raise RuntimeError("Failed to parse output for alpha={}, beta={}, gamma={}".format(alpha, beta, gamma))

# Iterate over all combinations
for alpha, beta, gamma in itertools.product(alpha_values, beta_values, gamma_values):
    print(f"Testing alpha={alpha}, beta={beta}, gamma={gamma}...")
    try:
        brrt_len, opt_len = run_test(alpha, beta, gamma)
        results.append((alpha, beta, gamma, brrt_len, opt_len))
    except Exception as e:
        print("Error:", e)




Testing alpha=0.1, beta=0.1, gamma=0.1...
Base path: /home/xuanloc/DACN/sampling-based-path-finding
Source space: /home/xuanloc/DACN/sampling-based-path-finding/src
Build space: /home/xuanloc/DACN/sampling-based-path-finding/build
Devel space: /home/xuanloc/DACN/sampling-based-path-finding/devel
Install space: /home/xuanloc/DACN/sampling-based-path-finding/install
####
#### Running command: "make cmake_check_build_system" in "/home/xuanloc/DACN/sampling-based-path-finding/build"
####
####
#### Running command: "make -j4 -l4" in "/home/xuanloc/DACN/sampling-based-path-finding/build"
####
[  0%] Built target std_msgs_generate_messages_eus
[  0%] Built target _self_msgs_and_srvs_generate_messages_check_deps_GlbObsRcv
[  0%] Built target _self_msgs_and_srvs_generate_messages_check_deps_input_point
[  0%] Built target _self_msgs_and_srvs_generate_messages_check_deps_output_point
[  0%] Built target geometry_msgs_generate_messages_eus
[  0%] Built target geometry_msgs_generate_messages_nodej

KeyboardInterrupt: 

In [ ]:
# Find best configuration where Optimize beats BRRT by largest margin
best = min(results, key=lambda r: r[3] - r[4])
alpha, beta, gamma, b_len, o_len = best



In [ ]:
print("\nBest parameters:")
print(f"  alpha={alpha}, beta={beta}, gamma={gamma}")
print(f"  BRRT length: {b_len}, Optimize length: {o_len}")
print(f"  Improvement: {b_len - o_len}")

# Optionally, write results to CSV
import csv
with open("brrt_opt_test_results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["alpha", "beta", "gamma", "brrt_length", "opt_length"])
    writer.writerows(results)